In [1]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
from torchvision.datasets import OxfordIIITPet

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224
BATCH_SIZE = 64
#Train transform,aug
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), # Slight cropping
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])
#validation transform
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])
train_dataset = OxfordIIITPet(root='./data', split='trainval', download=True, transform=train_transforms)
val_dataset = datasets.OxfordIIITPet(root='./data', split='test', target_types='category', download=True, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,pin_memory=True,drop_last=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,pin_memory=True)

print("\n-----------------Verification ---")
train_features, train_labels = next(iter(train_loader))

print(f"Feature batch shape: {train_features.shape}") # Expected: [64, 3, 224, 224]
print(f"Labels batch shape: {train_labels.shape}")    # Expected: [64]
print(f"Total training batches: {len(train_loader)}")
print(f"Number of classes: {len(train_dataset.classes)}")

100%|██████████| 792M/792M [00:45<00:00, 17.4MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 8.56MB/s]



-----------------Verification ---
Feature batch shape: torch.Size([64, 3, 224, 224])
Labels batch shape: torch.Size([64])
Total training batches: 57
Number of classes: 37


In [4]:
#image patching
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=192):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

img = torch.randn(1, 3, 224, 224)
patch_embed = PatchEmbedding()
print("Patch Embed Output Shape:", patch_embed(img).shape)
# Expected: torch.Size([1, 196, 192])

Patch Embed Output Shape: torch.Size([1, 196, 192])


In [5]:
class ViTEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=192, dropout=0.1):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches #196

        #CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        self.dropout = nn.Dropout(dropout)

        # Initialize weights (important for stable training)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_token, x], dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)

        return x



embed = ViTEmbedding()
img = torch.randn(4, 3, 224, 224)  # batch of 4
out = embed(img)
print("Embedding output:", out.shape)
# Expected: torch.Size([4, 197, 192])

Embedding output: torch.Size([4, 197, 192])


In [6]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3, attn_drop=0.1, proj_drop=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=True)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(proj_drop)
        assert embed_dim % num_heads == 0, \
          f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"

    def forward(self, x):
        B, N, D = x.shape

        #Project and split into Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        #Scaled Dot-Product Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # 3. Weighted sum over values
        out = attn @ v

        #Concatenate heads back together
        out = out.transpose(1, 2).reshape(B, N, D)

        #Final linear projection
        out = self.proj(out)
        out = self.proj_drop(out)
        return out

# Quick Shape Check
tokens = torch.randn(1, 197, 192) #+1 CLS TOKEN
mhsa = MultiHeadSelfAttention()
print("MHSA Output Shape:", mhsa(tokens).shape)
# Expected: torch.Size([1, 197, 192])

MHSA Output Shape: torch.Size([1, 197, 192])


In [8]:
class MLP(nn.Module):
    def __init__(self, in_features=192, hidden_features=None, out_features=None, drop=0.1):
        super().__init__()
        hidden_features = hidden_features or in_features * 4
        out_features = out_features or in_features

        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        # (B, N, D) -> (1, 196, 192)
        x = self.fc1(x) # Out: (1, 196, 768)
        x = self.act(x)
        x = self.drop(x)

        x = self.fc2(x) # Out:(1, 196, 192)
        x = self.drop(x)

        return x

tokens = torch.randn(1, 196, 192)
mlp_block = MLP(in_features=192)
print("MLP Output Shape:", mlp_block(tokens).shape)
# Expected:([1, 196, 192])

MLP Output Shape: torch.Size([1, 196, 192])


In [13]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3, mlp_ratio=4.0, drop=0.1, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, attn_drop=drop, proj_drop=drop)

        # This is the line that was crashing!
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_features = int(embed_dim * mlp_ratio)
        self.mlp = MLP(in_features=embed_dim, hidden_features=hidden_features, drop=drop)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

# Quick Shape Check
tokens = torch.randn(1, 196, 192) # B=1, N=196, D=192
transformer_block = TransformerBlock(embed_dim=192, num_heads=3)
print("Transformer Block Output Shape:", transformer_block(tokens).shape)
# Expected: torch.Size([1, 196, 192])

Transformer Block Output Shape: torch.Size([1, 196, 192])


In [14]:
def drop_path(x, drop_prob: float = 0., training: bool = False):
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # B, 1, 1
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()  # binarize
    return x.div(keep_prob) * random_tensor

class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

In [15]:
class ViT(nn.Module):
    def __init__(
        self, img_size=224, patch_size=16, in_channels=3, num_classes=37,
        embed_dim=192, num_heads=3, num_layers=12, mlp_ratio=4.0,
        drop=0.1, drop_path_max=0.1
    ):
        super().__init__()

        # Patch + positional embedding
        self.embedding = ViTEmbedding(
            img_size=img_size, patch_size=patch_size, in_channels=in_channels,
            embed_dim=embed_dim, dropout=drop
        )

        # Stacked transformer blocks
        drop_path_rates = torch.linspace(0, drop_path_max, num_layers).tolist()
        self.blocks = nn.Sequential(*[
            TransformerBlock(
                embed_dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,
                drop=drop, drop_path=drop_path_rates[i]
            )
            for i in range(num_layers)
        ])

        # Final LayerNorm
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head (only sees the CLS token)
        self.head = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        # ViT init
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.embedding(x)          # (B, 197, 192)
        x = self.blocks(x)             # (B, 197, 192)
        x = self.norm(x)               # (B, 197, 192)
        cls = x[:, 0]                  # (B, 192) grabs ONLY CLS token
        logits = self.head(cls)        # (B, 37)
        return logits


if __name__ == "__main__":
    print("Initializing ViT-Tiny model...")
    model = ViT(num_classes=37)
    dummy_img = torch.randn(4, 3, 224, 224)
    logits = model(dummy_img)

    print("Output logits shape:", logits.shape)   # Expected:([4, 37])

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params / 1e6:.2f}M")

Initializing ViT-Tiny model...
Output logits shape: torch.Size([4, 37])
Total parameters: 5.53M


In [16]:
import time
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
model = model.to(device)

Training on device: cuda


In [18]:
EPOCHS = 100
# ViTs need AdamW A learning rate of 1e-3 or 3e-4 is standard
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = GradScaler()

/tmp/ipykernel_1098/2978622944.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [19]:
best_val_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    # --- TRAINING PASS ---
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        # Scale loss and backpropagate
        scaler.scale(loss).backward()

        # Unscale before gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Optimizer step via scaler
        scaler.step(optimizer)
        scaler.update()

        # Track metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100. * val_correct / val_total

    epoch_time = time.time() - start_time

    # Print Epoch Results
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Time: {epoch_time:.0f}s")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%\n")

    # Save Best Model Checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_vit_tiny.pth')
        print("--> Saved new best checkpoint!\n")

print(f"Training Complete. Best Validation Accuracy: {best_val_acc:.2f}%")

/tmp/ipykernel_1098/1556649492.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1098/1556649492.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/100] | Time: 49s
Train Loss: 3.6468 | Train Acc: 3.78%
Val Loss:   3.5840 | Val Acc:   3.76%

--> Saved new best checkpoint!

Epoch [2/100] | Time: 48s
Train Loss: 3.5735 | Train Acc: 4.00%
Val Loss:   3.5743 | Val Acc:   4.52%

--> Saved new best checkpoint!

Epoch [3/100] | Time: 48s
Train Loss: 3.5570 | Train Acc: 4.85%
Val Loss:   3.5447 | Val Acc:   5.48%

--> Saved new best checkpoint!

Epoch [4/100] | Time: 48s
Train Loss: 3.5287 | Train Acc: 5.98%
Val Loss:   3.5188 | Val Acc:   4.58%

Epoch [5/100] | Time: 48s
Train Loss: 3.4929 | Train Acc: 6.91%
Val Loss:   3.5049 | Val Acc:   6.57%

--> Saved new best checkpoint!

Epoch [6/100] | Time: 46s
Train Loss: 3.4557 | Train Acc: 7.13%
Val Loss:   3.4447 | Val Acc:   7.33%

--> Saved new best checkpoint!

Epoch [7/100] | Time: 47s
Train Loss: 3.4464 | Train Acc: 8.28%
Val Loss:   3.4883 | Val Acc:   7.28%

Epoch [8/100] | Time: 46s
Train Loss: 3.4096 | Train Acc: 9.29%
Val Loss:   3.4870 | Val Acc:   7.71%

--> Saved new be